# Experiment Data Analysis

Produces figures and tables for the classroom inflation forecasting experiment section of the paper. Output files are written to `../img/`, `../figures/`, and `../tables/`.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import fredpy as fp
import automate_teaching as at

# Plot style
plt.style.use('classic')
plt.rcParams['figure.facecolor'] = 'white'

# Output directories (relative to this notebook)
img_dir   = '../img'
fig_dir   = '../figures'
table_dir = '../tables'

# Helpers that turn a date index into the range text used in figure and table captions,
# so the captions always match the data actually plotted or tabulated.
def year_range(idx):
    idx = pd.DatetimeIndex(idx)
    return f'{idx.min().year}--{idx.max().year}'

def quarter_range(idx):
    idx = pd.DatetimeIndex(idx)
    return f'{idx.min().year}:Q{idx.min().quarter}--{idx.max().year}:Q{idx.max().quarter}'

def month_range(idx):
    idx = pd.DatetimeIndex(idx)
    return f'{idx.min():%B %Y}--{idx.max():%B %Y}'

## 2. Survey Forecast Data

Inflation forecasts from two surveys — the Survey of Professional Forecasters (SPF) and the University of Michigan Surveys of Consumers — alongside the corresponding realized inflation series.

**Timing convention.** The CSV files in `../csv/` are dated by when the forecast was made (origin-dated), so that `interest - forecast` is the ex ante real rate on any row. For comparisons with realized inflation and with the experiment, every series in this notebook is instead dated by the period being forecast (target-dated), the way the student rounds are: the SPF series are shifted forward one year (four quarters) and the Michigan forecast is shifted twelve months. The one exception is the real interest rate figure (Section 2.6), which uses the CSV dating on purpose.

### 2.1 Actual GDP deflator inflation

Annual GDP deflator inflation from FRED (`A191RD3A086NBEA`), computed as the year-over-year percentage change *ending* in each year (target-dated: the value at year $t$ is growth from $t-1$ to $t$) and restricted to 1971--1990, the target years of the SPF forecasts made in 1970--1989.

In [ ]:
# Actual GDP deflator inflation, TARGET-dated: the value at year t is realized growth from
# year t-1 to year t, i.e. the outcome of the forecast made during year t-1 for year t. This
# is how the student rounds are dated (round t forecasts year t). The CSV files are
# origin-dated (see inflation_forecast_data.ipynb), so SPF series read from them are
# shifted forward one year below.
gdp_def = fp.series('A191RD3A086NBEA')
inflation = 100 * (gdp_def.data / gdp_def.data.shift(1) - 1)
inflation = inflation.loc['1971':'1990']

### 2.2 SPF: forecast versus actual (quarterly)

Median one-year-ahead GDP deflator inflation forecast from the SPF, plotted against realized inflation. The CSV is origin-dated, so both columns are shifted forward four quarters (one year) here: each point shows the forecast made a year earlier next to the inflation realized over the four quarters ending on that date.

In [ ]:
spf_quarterly = pd.read_csv(
    '../csv/inflation_forecast_data_quarterly.csv',
    index_col=0, parse_dates=True
)

# The CSV is origin-dated (forecast made at t, inflation from t to t+4). Re-date both columns
# to the end of the forecast horizon (t + 4 quarters) so that each point is the forecast made
# a year earlier next to the inflation realized over the four quarters ending on that date.
spf_quarterly.index = spf_quarterly.index + pd.DateOffset(years=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(spf_quarterly['deflator inflation - actual'],   '-',  lw=5, color='#1f77b4', label='Actual inflation')
ax.plot(spf_quarterly['deflator inflation - forecast'], '--', lw=3, color='#ff7f0e', label='Median forecast')
ax.set_ylabel('Inflation rate (%)')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=2)
fig.autofmt_xdate()
fig.tight_layout()

fig_name = 'spf_forecast_actual'
caption = (
    'Median one-year-ahead GDP deflator inflation forecast from the Survey of Professional '
    'Forecasters and actual U.S. inflation (percent change in the GDP deflator over the '
    'previous four quarters). Quarterly, ' + quarter_range(spf_quarterly.dropna().index) + '. Forecast values are lagged four quarters to align '
    'with the forecast horizon. Source: Survey of Professional Forecasters and FRED.'
)

plt.savefig(img_dir + '/fig_' + fig_name + '.png', bbox_inches='tight', dpi=120)

# Paper version
at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption=caption,
    label='fig:' + fig_name,
    filename=fig_dir + '/figure_' + fig_name + '.tex'
)

# Slides version (shorter, to fit a Beamer frame)
at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption=caption,
    label='fig:' + fig_name,
    height='5cm',
    filename=fig_dir + '/figure_' + fig_name + '_slides.tex'
);

### 2.3 Michigan: forecast versus actual (monthly)

Median expected price change over the next 12 months from the University of Michigan Surveys of Consumers (FRED series `MICH`, available from January 1978), plotted against realized CPI inflation (`CPIAUCSL`).

The Michigan series is shifted forward twelve months so that each observation pairs a forecast with the twelve-month inflation rate it was predicting. Because `MICH` begins in January 1978, the aligned series begins in January 1979. The resulting DataFrame `michigan_monthly` is reused in Section 5.1.

In [ ]:
# Forecast: shift forward 12 months so each value sits at the date of the outcome it predicted
michigan_forecast = fp.series('MICH').data.shift(12)
michigan_forecast.name = 'cpi inflation - forecast'

# Actual: twelve-month percent change in the CPI
cpi = fp.series('CPIAUCSL')
cpi_inflation = 100 * (cpi.data / cpi.data.shift(12) - 1)
cpi_inflation = cpi_inflation.dropna()
cpi_inflation.name = 'cpi inflation - actual'

# Aligned forecast-outcome pairs; drops the pre-1979 rows with no matching forecast
michigan_monthly = pd.concat([michigan_forecast, cpi_inflation], axis=1).dropna()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(michigan_monthly['cpi inflation - actual'],   '-',  lw=5, color='#1f77b4', label='Actual inflation')
ax.plot(michigan_monthly['cpi inflation - forecast'], '--', lw=3, color='#ff7f0e', label='Median forecast')
ax.set_ylabel('Inflation rate (%)')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=2)
fig.autofmt_xdate()
fig.tight_layout()

fig_name = 'michigan_forecast_actual'
caption = (
    'Median one-year-ahead inflation forecast from the University of Michigan Survey of '
    'Consumers and actual U.S. inflation (percent change in the CPI over the previous 12 '
    'months). Monthly, ' + month_range(michigan_monthly.index) + '. Forecast values are lagged 12 months to align with the forecast '
    'horizon. Source: University of Michigan Survey of Consumers and FRED.'
)

plt.savefig(img_dir + '/fig_' + fig_name + '.png', bbox_inches='tight', dpi=120)

# Paper version
at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption=caption,
    label='fig:' + fig_name,
    filename=fig_dir + '/figure_' + fig_name + '.tex'
)

# Slides version (shorter, to fit a Beamer frame)
at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption=caption,
    label='fig:' + fig_name,
    height='5cm',
    filename=fig_dir + '/figure_' + fig_name + '_slides.tex'
);

### 2.4 Individual SPF forecasts

Four-quarter-ahead GDP deflator inflation forecasts from the SPF individual microdata. Quarterly observations are averaged to annual frequency for each forecaster, then re-dated to the year being forecast (survey year + 1) and restricted to target years 1971--1990.

In [ ]:
try:
    data_individual = pd.read_excel('../xlsx/Individual_PGDP.xlsx', sheet_name='PGDP')
except FileNotFoundError:
    print('Individual forecast data not found locally — downloading from Philadelphia Fed.')
    data_individual = pd.read_excel(
        'https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/survey-of-professional-forecasters/data-files/files/Individual_PGDP.xlsx?sc_lang=en&hash=1F457E241D7AEF1C453EB13DB3968D07',
        sheet_name='PGDP'
    )

# Four-quarter-ahead forecast: percentage change from current-quarter level (PGDP2) to four quarters ahead (PGDP6)
data_individual['deflator inflation - forecast'] = 100 * (data_individual['PGDP6'] / data_individual['PGDP2'] - 1)

# Convert YEAR to datetime for consistent index handling
# Date each forecast by the year being forecast (survey year + 1), matching the target-dated
# actual inflation series and the student rounds
data_individual['YEAR'] = pd.to_datetime(data_individual['YEAR'], format='%Y') + pd.DateOffset(years=1)

# Average quarterly observations to annual, preserving forecaster identity in MultiIndex
spf_individual = (
    data_individual
    .groupby(['YEAR', 'ID'])['deflator inflation - forecast']
    .mean()
    .to_frame()
)
spf_individual = spf_individual.loc['1971':'1990']

# Cross-sectional median and standard deviation by year
spf_median   = spf_individual.groupby('YEAR')['deflator inflation - forecast'].median()
spf_std_devs = spf_individual.groupby('YEAR')['deflator inflation - forecast'].std()

### 2.5 SPF forecast dispersion figures

In [ ]:
# Figure: median SPF forecast with ±1 SD band
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(inflation,  color='#1f77b4', linewidth=5, marker='o', markersize=6, label='Actual inflation')
ax.plot(spf_median, color='#ff7f0e', linewidth=3, marker='o', markersize=6,
        linestyle='--', label='Median forecast')
ax.fill_between(spf_median.index, spf_median - spf_std_devs, spf_median + spf_std_devs,
                alpha=0.15, color='#ff7f0e', label='±1 SD of forecasts')
ax.set_ylabel('Inflation rate (%)', fontsize=11)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncols=3)
fig.autofmt_xdate()
fig.tight_layout()

fig_name = 'spf_median_std'
plt.savefig(img_dir + '/fig_' + fig_name + '.png', bbox_inches='tight', dpi=120)
at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption='Actual U.S. GDP deflator inflation and the median SPF one-year-ahead forecast, ' + year_range(spf_median.index) + '. '
            'Shaded band indicates $\\pm1$ standard deviation of individual forecasts. The SPF series extends to 1990, one year beyond the student rounds, because SPF forecasts made in 1989 are for 1990.',
    label='fig:' + fig_name,
    filename=fig_dir + '/figure_' + fig_name + '.tex'
)

at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption='Actual U.S. GDP deflator inflation and the median SPF one-year-ahead forecast, ' + year_range(spf_median.index) + '. '
            'Shaded band indicates $\\pm1$ standard deviation of individual forecasts. The SPF series extends to 1990, one year beyond the student rounds, because SPF forecasts made in 1989 are for 1990.',
    label='fig:' + fig_name,
    height='5cm',
    filename=fig_dir + '/figure_' + fig_name + '_slides.tex'
)

In [ ]:
# Figure: individual SPF forecasts overlaid on actual inflation
fig, ax = plt.subplots(figsize=(10, 5))
for forecaster_id, group in spf_individual.groupby('ID'):
    ax.plot(group.index.get_level_values('YEAR'), group['deflator inflation - forecast'],
            linewidth=0.5, color='black', alpha=0.15)
ax.plot(inflation, color='#1f77b4', linewidth=5, marker='o', markersize=6, label='Actual inflation')
ax.plot([], [], linewidth=0.3, color='black', alpha=0.5, label='Individual forecasts')
ax.set_ylim([0, 12])
ax.set_ylabel('Inflation rate (%)', fontsize=11)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncols=3)
fig.autofmt_xdate()
fig.tight_layout()

fig_name = 'spf_median_individual'
plt.savefig(img_dir + '/fig_' + fig_name + '.png', bbox_inches='tight', dpi=120)
at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption='Individual SPF one-year-ahead forecasts of GDP deflator inflation, ' + year_range(spf_individual.index.get_level_values('YEAR')) + ', '
            'alongside actual inflation. Each grey line represents a single forecaster. The SPF series extends to 1990, one year beyond the student rounds, because SPF forecasts made in 1989 are for 1990.',
    label='fig:' + fig_name,
    filename=fig_dir + '/figure_' + fig_name + '.tex'
)

plt.savefig(img_dir + '/fig_' + fig_name + '.png', bbox_inches='tight', dpi=120)
at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption='Individual SPF one-year-ahead forecasts of GDP deflator inflation, ' + year_range(spf_individual.index.get_level_values('YEAR')) + ', '
            'alongside actual inflation. Each grey line represents a single forecaster. The SPF series extends to 1990, one year beyond the student rounds, because SPF forecasts made in 1989 are for 1990.',
    label='fig:' + fig_name,
    height='5cm',
    filename=fig_dir + '/figure_' + fig_name + '_slides.tex'
)

### 2.6 Aggregate SPF data figures

These figures use the aggregate annual SPF dataset downloaded directly from the Philadelphia Fed.

In [ ]:
annual_data = pd.read_csv('../csv/inflation_forecast_data_annual.csv', index_col=0, parse_dates=True)  # origin-dated, as in the CSV (used unchanged for the real-rate figure below)

# Re-date to the year being forecast: row t of the CSV holds the forecast made during year t
# and inflation from t to t+1, so shift forward one year. Only the forecast and actual columns
# are kept; the interest rate column is not used in this figure.
annual_target = annual_data[['deflator inflation - forecast', 'deflator inflation - actual']].copy()
annual_target.index = annual_target.index + pd.DateOffset(years=1)

# Consistency check: the target-dated actual inflation computed in Section 2.1 from FRED should
# match the CSV's actual column shifted forward one year. A large difference means the CSV is stale.
_common = annual_target.index.intersection(inflation.index)
print('Max |CSV actual (shifted) - FRED actual|:', (annual_target.loc[_common, 'deflator inflation - actual'] - inflation.loc[_common]).abs().max())

# Annual: actual vs. forecast inflation with shading
# Blue shading: actual inflation exceeded forecast (π > Eπ)
# Red shading: actual inflation fell short of forecast (π < Eπ)
# Same figure as above, but with annual data
fig,ax = plt.subplots(figsize=(10,5))
# fig,ax = plt.subplots()
ax.plot(annual_target['deflator inflation - actual'],'-',color='#1f77b4',lw=5,label='Actual inflation')
ax.plot(annual_target['deflator inflation - forecast'],'--',color='#ff7f0e',lw=4,label='Median forecast')
ax.fill_between(annual_target.index,annual_target['deflator inflation - forecast'],np.max([annual_target['deflator inflation - actual'],annual_target['deflator inflation - forecast']],axis=0),color='b',alpha=0.15,label='Actual above forecast')
ax.fill_between(annual_target.index,np.max([annual_target['deflator inflation - actual'],annual_target['deflator inflation - forecast']],axis=0),annual_target['deflator inflation - actual'],color='r',alpha=0.5,label='Actual below forecast')
fp.recessions(start=annual_target.index[0],end=annual_target.index[-1])
ax.grid()
ax.set_ylabel('Inflation rate (%)')

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncols=2)
fig.autofmt_xdate()
fig.tight_layout()


fig_name = 'us_inflation_forecast_annual'
plt.savefig(img_dir + '/fig_' + fig_name + '.png', bbox_inches='tight', dpi=120)
at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption='Median SPF one-year-ahead forecast and actual GDP deflator inflation, ' + year_range(annual_target.dropna().index) + '. Blue (red) shading indicates years in which actual inflation exceeded (fell short of) the median '
            'forecast. Each year shows the forecast made in the previous year for that year and the actual inflation over that year. Shaded vertical bars denote NBER recessions.',
    label='fig:' + fig_name,
    height='6.5cm',
    filename=fig_dir + '/figure_' + fig_name + '.tex'
)

at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption='Median SPF one-year-ahead forecast and actual GDP deflator inflation, ' + year_range(annual_target.dropna().index) + '. Blue (red) shading indicates years in which actual inflation exceeded (fell short of) the median '
            'forecast. Each year shows the forecast made in the previous year for that year and the actual inflation over that year. Shaded vertical bars denote NBER recessions.',
    label='fig:' + fig_name,
    height='5.cm',
    filename=fig_dir + '/figure_' + fig_name + '_slides.tex'
);

In [ ]:
# NOTE: this figure deliberately uses the origin-dated CSV (annual_data, not annual_target):
# the interest rate and the forecast made in the same year give the ex ante real rate i_t - E_t[pi_{t+1}].
# Annual: two-panel figure showing (1) nominal rate and expected inflation,
# and (2) the implied ex ante real interest rate (i - Eπ)
fig = plt.figure(figsize=[10,10])
ax = fig.add_subplot(2,1,1)
ax.plot(annual_data['nominal interest'],color='#1f77b4',label='Nominal interest rate',lw=5)
ax.plot(annual_data['deflator inflation - forecast'],'--',color='#ff7f0e',label='Median forecast',lw=4)
fp.recessions(start=annual_data.index[0],end=annual_data.index[-1])
ax.set_ylabel('Percent')
ax.grid()

ax.legend(loc='upper right',ncol=1,fontsize=13)

fig.autofmt_xdate()
fig.tight_layout()

ax = fig.add_subplot(2,1,2)
ax.plot(annual_data['nominal interest']-annual_data['deflator inflation - forecast'],'k',lw=5,label='Ex ante real interest rate')
fp.recessions(start=annual_data.index[0],end=annual_data.index[-1])
ax.set_ylabel('Percent')
ax.grid()
ax.legend(loc='upper right',ncol=1,fontsize=13)

fig_name = 'us_real_interest_rate_annual'
real_rate = (annual_data['nominal interest'] - annual_data['deflator inflation - forecast']).dropna()
plt.savefig(img_dir + '/fig_' + fig_name + '.png', bbox_inches='tight', dpi=120)
at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption='The \\emph{ex ante} real interest rate, ' + year_range(real_rate.index) + '. Each year is the year in which the forecast was made. Top panel: the nominal interest rate and the median SPF one-year-ahead '
            'inflation forecast. Bottom panel: the implied \\emph{ex ante} real interest rate, computed as the nominal rate less the '
            'median forecast.',
    label='fig:' + fig_name,
    height='13cm',
    filename=fig_dir + '/figure_' + fig_name + '.tex'
);

## 3. Experiment Data

### 3.1 Load and summarize

Load the anonymized results CSV and compute per-round summary statistics. All 20 rounds are included (rounds 1 & 2 are practice and unscored). Forecasts recorded as timeouts (`is_timeout == 1`) are not real forecasts: the recorded value is exactly 0.0. They are excluded from every forecast statistic and figure below (the survey responses in Section 4 still use all participants).

In [ ]:
experimental_results = pd.read_csv('../experiment_results/inflation_forecast_results_anonymized.csv')

# Datetime index aligning round numbers to calendar years 1970--1989
index = pd.date_range(start='1970-01-01', end='1989-01-01', freq='YS')

# Drop timed-out rounds: the recorded forecast is a default 0.0, not a submitted forecast
n_timeout = int(experimental_results['is_timeout'].sum())
n_students_timeout = experimental_results.loc[experimental_results['is_timeout'] == 1, 'participant_id'].nunique()
scored  = experimental_results[experimental_results['is_timeout'] == 0]
print(f'Excluded {n_timeout} timed-out forecasts from {n_students_timeout} students; '
      f'{len(scored)} submitted forecasts remain.')
print('Submitted forecasts at the 15% bound:', int((scored['forecast'] == 15).sum()),
      '| at the -5% bound:', int((scored['forecast'] == -5).sum()))
actuals = scored.groupby('round_number')['actual_inflation'].first()
means   = scored.groupby('round_number')['forecast'].mean()
medians   = scored.groupby('round_number')['forecast'].median()
stds    = scored.groupby('round_number')['forecast'].std()

actuals.index = index
means.index   = index
medians.index   = index
stds.index    = index

### 3.2 Experiment figures

In [ ]:
# Figure: median student forecast with ±1 SD band
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(actuals, color='#1f77b4', linewidth=5, marker='o', markersize=6, label='Actual inflation')
ax.plot(medians,   color='#ff7f0e', linewidth=3, marker='o', markersize=6,
        linestyle='--', label='Median forecast')
ax.fill_between(actuals.index, medians - stds, medians + stds,
                alpha=0.15, color='#ff7f0e', label='±1 SD of forecasts')
ax.set_ylabel('Inflation rate (%)', fontsize=11)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncols=3)
fig.autofmt_xdate()
fig.tight_layout()

fig_name = 'class_median_std'
plt.savefig(img_dir + '/fig_' + fig_name + '.png', bbox_inches='tight', dpi=120)
at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption='Median student forecast of PCE deflator inflation and actual inflation by round, ' + year_range(actuals.index) + '. '
            'Shaded band indicates $\\pm1$ standard deviation of student forecasts.',
    label='fig:' + fig_name,
    filename=fig_dir + '/figure_' + fig_name + '.tex'
)

at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption='Median student forecast of PCE deflator inflation and actual inflation by round, ' + year_range(actuals.index) + '. '
            'Shaded band indicates $\\pm1$ standard deviation of student forecasts.',
    label='fig:' + fig_name,
    height='5cm',
    filename=fig_dir + '/figure_' + fig_name + '_slides.tex'
)

In [ ]:
# Figure: individual student forecasts overlaid on actual inflation
fig, ax = plt.subplots(figsize=(10, 5))
for participant_id in experimental_results['participant_id'].unique():
    p_rows = scored[scored['participant_id'] == participant_id].sort_values('round_number')
    # plot each student at the dates of the rounds actually submitted (timed-out rounds are dropped)
    ax.plot(actuals.index[p_rows['round_number'].astype(int) - 1], p_rows['forecast'], linewidth=0.5, color='black', alpha=0.15)
ax.plot(actuals, color='#185FA5', linewidth=5, marker='o', markersize=6,
        label='Actual inflation', zorder=5)
ax.plot([], [], linewidth=0.3, color='black', alpha=0.5, label='Individual forecasts')
ax.set_ylabel('Inflation rate (%)', fontsize=11)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncols=3)
fig.autofmt_xdate()
fig.tight_layout()

fig_name = 'class_median_individual'
plt.savefig(img_dir + '/fig_' + fig_name + '.png', bbox_inches='tight', dpi=120)
at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption='Individual student forecasts of PCE deflator inflation by round, ' + year_range(actuals.index) + ', '
            'alongside actual inflation. Each grey line represents a single student.',
    label='fig:' + fig_name,
    filename=fig_dir + '/figure_' + fig_name + '.tex'
)

at.latex.make_figure(
    image_filename='img/fig_' + fig_name + '.png',
    caption='Individual student forecasts of PCE deflator inflation by round, ' + year_range(actuals.index) + ', '
            'alongside actual inflation. Each grey line represents a single student.',
    label='fig:' + fig_name,
    height='5cm',
    filename=fig_dir + '/figure_' + fig_name + '_slides.tex'
)

## 4. End-of-Experiment Survey

Survey responses are drawn from round 20, the final round.

### 4.1 Effort ratings

In [ ]:
effort_labels = {
    1: 'I mostly clicked through without thinking',
    2: 'Some effort, but not much',
    3: 'Moderate effort',
    4: 'Fairly serious effort',
    5: 'I tried hard to forecast as accurately as possible'
}

round_20 = experimental_results[experimental_results['round_number'] == 20]
counts = round_20['effort_rating'].value_counts().sort_index()
pcts   = counts / counts.sum() * 100

effort_array = np.array(
    [['Rating', 'Description', 'Percent']] +
    [[str(int(k)), effort_labels[k], f'{pcts[k]:.1f}'] for k in sorted(effort_labels)]
)

tab_name = 'effort_rating'
at.latex.make_table(
    data=effort_array,
    table_spec='clc',
    hlines=[1],
    label='tab:' + tab_name,
    caption="Self-reported effort ratings. Students were asked ``How much effort did you put into making accurate forecasts?'' at the end of the experiment.",
    filename=table_dir + '/table_' + tab_name + '.tex'
)

# Table note: number of respondents (some participants skipped the question)
n_answered = int(counts.sum())
n_skipped  = len(round_20) - n_answered
number_words = {1:'one',2:'two',3:'three',4:'four',5:'five',6:'six',7:'seven',8:'eight',9:'nine'}
effort_note = (f'Percentages are of the {n_answered} participants who answered; '
               f'{number_words.get(n_skipped, n_skipped)} did not.')
tab_file = table_dir + '/table_' + tab_name + '.tex'
with open(tab_file) as f:
    tex = f.read()
tex = tex.replace('\\end{table}', '\\footnotesize ' + effort_note + '\n\\end{table}')
with open(tab_file, 'w') as f:
    f.write(tex)

### 4.2 Indicators used

In [ ]:
indicator_labels = {
    'lag_inflation_only': 'Lagged inflation only',
    'all_three':          r'All three indicators\textsuperscript{$a$}',
    'inflation_and_one':  'Lagged inflation plus one other indicator',
    'other':              'Some other approach'
}

round_20 = experimental_results[experimental_results['round_number'] == 20]
counts = round_20['used_indicators'].value_counts()
pcts   = counts / counts.sum() * 100

indicator_array = np.array(
    [['Description', 'Percent']] +
    [[indicator_labels[k], f'{pcts[k]:.1f}'] for k in indicator_labels]
)

tab_name = 'used_indicators'
at.latex.make_table(
    data=indicator_array,
    table_spec='lc',
    hlines=[1],
    label='tab:' + tab_name,
    caption="Self-reported information use. Students were asked ``Which information did you primarily use when forming your forecasts?'' at the end of the experiment.",
    filename=table_dir + '/table_' + tab_name + '.tex'
)

## 5. SPF vs. Student Comparison

Forecast performance is evaluated at the **median forecast** level for every group: for each year (SPF, Michigan) or round (students), the median forecast across all participants is compared to realized inflation. Evaluating at the median keeps the comparison symmetric with the naive benchmark, which is itself a single series, and avoids folding cross-sectional disagreement into the accuracy measure. Dispersion across individual forecasters is shown separately in the figures of Sections 2 and 3.

**Sign convention.** Forecast error is defined throughout as the forecast minus realized inflation, $\pi^f_t - \pi_t$, the same convention as the experiment's feedback screen. A positive mean error therefore means the group over-forecast inflation on average.

**Normalized measures.** One normalization makes the groups comparable despite their different target series. **RMSE\/SD** divides RMSE by the standard deviation of the group's own realized inflation series over the same sample.

**Window.** Table 1 uses the forecast target years common to both groups, 1971--1989 (19 years). The SPF target years are 1971--1990 (forecasts made in 1970--1989) and the student rounds cover 1970--1989. Change `table_start` and `table_end` in the next code cell to use a different window.

In [ ]:
# Table 1 window: forecast target years common to both groups.
# SPF target years are 1971--1990 (forecasts made in 1970--1989); student rounds cover 1970--1989.
table_start, table_end = '1971', '1989'

# ── SPF: median forecast vs. realized GDP deflator inflation ─────────────────
spf_inflation_w = inflation.loc[table_start:table_end]
spf_median_w    = spf_median.loc[table_start:table_end]
# Forecast error = forecast minus realized inflation (positive = over-forecast), as on the feedback screen
spf_median_error = spf_median_w - spf_inflation_w

spf_bias    = spf_median_error.mean()
spf_rmse    = (spf_median_error**2).mean()**0.5
spf_rmse_sd = spf_rmse / spf_inflation_w.std()
spf_corr    = spf_median_w.corr(spf_inflation_w)

# Naive benchmark: prior year's realized inflation
spf_naive_rmse = ((spf_inflation_w - spf_inflation_w.shift(1))**2).mean()**0.5
spf_theil_u    = spf_rmse / spf_naive_rmse

# ── Students: median forecast vs. realized inflation by round ────────────────
student_median   = medians  # full series, computed in Section 3.1 (also used in Section 5.1)
student_median_w = student_median.loc[table_start:table_end]
actuals_w        = actuals.loc[table_start:table_end]
student_median_error = student_median_w - actuals_w

student_bias    = student_median_error.mean()
student_rmse    = (student_median_error**2).mean()**0.5
student_rmse_sd = student_rmse / actuals_w.std()
student_corr    = student_median_w.corr(actuals_w)

# Naive benchmark: prior round's realized inflation
student_naive_rmse = ((actuals_w - actuals_w.shift(1))**2).mean()**0.5
student_theil_u    = student_rmse / student_naive_rmse

# Window used in the Table 1 caption; the two groups must cover the same years
assert spf_inflation_w.index.equals(actuals_w.index), 'SPF and student windows differ'
table_range = year_range(spf_inflation_w.index)
print('Table 1 window:', table_range, f'({len(spf_inflation_w)} years)')

In [ ]:
# Original two-column table (SPF vs. students, 1971-1989)
df = pd.DataFrame()
df.loc['Mean error (pp)',    'SPF']      = spf_bias
df.loc['Mean error (pp)',    'Students'] = student_bias
df.loc['RMSE (pp)',          'SPF']      = spf_rmse
df.loc['RMSE (pp)',          'Students'] = student_rmse
df.loc['RMSE / SD',          'SPF']      = spf_rmse_sd
df.loc['RMSE / SD',          'Students'] = student_rmse_sd
df.loc['Correlation',        'SPF']      = spf_corr
df.loc['Correlation',        'Students'] = student_corr
df

In [ ]:
data = at.latex.DataFrame_to_array(df.round(2).astype(str))

tab_name = 'stat_comparison'
tab_file = table_dir + '/table_' + tab_name + '.tex'
at.latex.make_table(
    data=data,
    hlines=[1],
    table_spec='lcc',
    label='tab:' + tab_name,
    caption='Forecast performance of the median SPF forecast and the median student forecast, '
            + table_range + '.',
    filename=tab_file
)

# Table note, placed below the table
table_note = (
    'Notes: The sample is the ' + str(len(spf_inflation_w)) + ' forecast target years common '
    'to both groups. Both groups forecast inflation in year $t$, one year ahead. Students saw '
    'complete, revised data through year $t-1$, while SPF panelists forecast during year $t-1$ '
    'with the data available at the time. All statistics are computed from the median forecast '
    'across participants in each year (SPF) or round (students). Forecast error is the forecast '
    'minus actual inflation, so a positive mean error indicates over-forecasting on average. '
    'Mean error and RMSE are in percentage points. RMSE / SD divides RMSE by the standard '
    "deviation of the group's own actual inflation over the same years. SPF forecasts are of "
    'GDP deflator inflation and student forecasts are of PCE deflator inflation, so RMSE / SD '
    'is the more appropriate basis for comparison across columns.'
)
with open(tab_file) as f:
    tex = f.read()
tex = tex.replace('\\end{table}', '\\begin{minipage}{0.9\\textwidth}\\footnotesize ' + table_note + '\\end{minipage}\n\\end{table}')
with open(tab_file, 'w') as f:
    f.write(tex)


### 5.1 Extended comparison with the Michigan Survey

The Michigan series (`MICH`) begins in January 1978, and the twelve-month alignment applied in Section 2.3 pushes the first usable observation to January 1979. All three groups are therefore restricted to **1979--1989** so that the columns describe a common sample.

Raw mean error and RMSE are not directly comparable across columns, since each group forecasts a different price index — the GDP deflator (SPF), the CPI (Michigan), and the PCE deflator (students). RMSE\/SD is the more appropriate basis for comparison.

In [ ]:
# Common sample: set by Michigan availability after the 12-month alignment
sample_start, sample_end = '1979', '1989'

# ── Michigan: aggregate the aligned monthly pairs to annual frequency ────────
# michigan_monthly already carries the 12-month shift, so averaging within a
# calendar year pairs forecasts made during the prior year with the inflation
# rates those forecasts were predicting.
michigan_annual = michigan_monthly.resample('YS').mean().loc[sample_start:sample_end]

mich_forecast = michigan_annual['cpi inflation - forecast']
mich_actual   = michigan_annual['cpi inflation - actual']

mich_error   = mich_forecast - mich_actual   # forecast minus actual (positive = over-forecast)
mich_bias    = mich_error.mean()
mich_rmse    = (mich_error**2).mean()**0.5
mich_rmse_sd = mich_rmse / mich_actual.std()
mich_corr    = mich_forecast.corr(mich_actual)

mich_naive_rmse = ((mich_actual - mich_actual.shift(1))**2).mean()**0.5
mich_theil_u    = mich_rmse / mich_naive_rmse

# ── SPF on the restricted sample ─────────────────────────────────────────────
inflation_sub  = inflation.loc[sample_start:sample_end]
spf_median_sub = spf_median.loc[sample_start:sample_end]

spf_error_sub      = spf_median_sub - inflation_sub
spf_bias_sub       = spf_error_sub.mean()
spf_rmse_sub       = (spf_error_sub**2).mean()**0.5
spf_rmse_sd_sub    = spf_rmse_sub / inflation_sub.std()
spf_naive_rmse_sub = ((inflation_sub - inflation_sub.shift(1))**2).mean()**0.5
spf_theil_u_sub    = spf_rmse_sub / spf_naive_rmse_sub
spf_corr_sub       = spf_median_sub.corr(inflation_sub)

# ── Students on the restricted sample ────────────────────────────────────────
actuals_sub        = actuals.loc[sample_start:sample_end]
student_median_sub = student_median.loc[sample_start:sample_end]

student_error_sub      = student_median_sub - actuals_sub
student_bias_sub       = student_error_sub.mean()
student_rmse_sub       = (student_error_sub**2).mean()**0.5
student_rmse_sd_sub    = student_rmse_sub / actuals_sub.std()
student_naive_rmse_sub = ((actuals_sub - actuals_sub.shift(1))**2).mean()**0.5
student_theil_u_sub    = student_rmse_sub / student_naive_rmse_sub
student_corr_sub       = student_median_sub.corr(actuals_sub)

In [ ]:
# Assemble extended DataFrame
df_mich = pd.DataFrame()
df_mich.loc['Mean error (pp)', 'SPF']      = spf_bias_sub
df_mich.loc['Mean error (pp)', 'Michigan'] = mich_bias
df_mich.loc['Mean error (pp)', 'Students'] = student_bias_sub
df_mich.loc['RMSE (pp)',       'SPF']      = spf_rmse_sub
df_mich.loc['RMSE (pp)',       'Michigan'] = mich_rmse
df_mich.loc['RMSE (pp)',       'Students'] = student_rmse_sub
df_mich.loc['RMSE / SD',       'SPF']      = spf_rmse_sd_sub
df_mich.loc['RMSE / SD',       'Michigan'] = mich_rmse_sd
df_mich.loc['RMSE / SD',       'Students'] = student_rmse_sd_sub
df_mich.loc['Correlation',     'SPF']      = spf_corr_sub
df_mich.loc['Correlation',     'Michigan'] = mich_corr
df_mich.loc['Correlation',     'Students'] = student_corr_sub
df_mich

In [ ]:
data_mich = at.latex.DataFrame_to_array(df_mich.round(2).astype(str))

tab_name = 'stat_comparison_michigan'
at.latex.make_table(
    data=data_mich,
    hlines=[1],
    table_spec='lccc',
    label='tab:' + tab_name,
    caption='Forecast performance of the median SPF forecast, the median Michigan Survey '
            'forecast, and the median student forecast, ' + year_range(michigan_annual.index) + '. '
            'All three columns cover the same sample. All three are one-year-ahead forecasts: '
            'each group observes data through year $t-1$ (or the prior twelve months, for '
            'Michigan) and forecasts the year (or twelve-month period) ahead. '
            'All statistics are computed from the median forecast across participants in each '
            'year (SPF, Michigan) or round (students), with each forecast aligned to the period '
            'it predicts. '
            'The sample begins in 1979 because the Michigan series starts in January 1978 and is '
            'lagged twelve months to match its forecast horizon. '
            'Forecast error is defined as the forecast minus actual inflation, so a positive '
            'mean error indicates over-forecasting on average. '
            'Mean error and RMSE are in percentage points. '
            'RMSE\/SD divides RMSE by the standard deviation of the group\'s own actual '
            'inflation series over the same sample. '
            'Each group forecasts a different price index, the GDP deflator (SPF), the CPI '
            '(Michigan), and the PCE deflator (students), so the mean error and RMSE rows are '
            'not strictly comparable across columns; RMSE\/SD is the more appropriate '
            'basis for comparison.',
    filename=table_dir + '/table_' + tab_name + '.tex'
)

## 6. Screenshot Figures

Generate LaTeX figure files for the experiment interface screenshots. PNG files must be placed in `screenshots/` relative to `main.tex`.

In [ ]:
screenshots = [
    (
        'screenshot_instructions_1',
        'Instructions screen (1 of 2) shown to participants at the start of the experiment. '
        'Students were informed of the three macroeconomic indicators available, the 20-round '
        'sequential structure, the forecasting bounds ($-5\\%$ to $15\\%$), the MSE scoring rule, '
        'and that grades were not affected by forecasting performance.'
    ),
    (
        'screenshot_decision',
        'Decision screen shown to participants at the start of each round. Students were presented '
        'with a table of recent macroeconomic indicators and a chart showing actual inflation alongside '
        'their own prior forecasts, then asked to enter a forecast for the upcoming year.'
    ),
    (
        'screenshot_feedback',
        'Feedback screen shown to participants immediately after each forecast. Students were shown '
        'their forecast, the actual inflation outcome, the forecast error, the squared error, whether '
        'they over- or under-forecasted, and their running MSE across all scored rounds to date.'
    ),
]

for fig_name, caption in screenshots:
    at.latex.make_figure(
        image_filename='screenshots/' + fig_name + '.png',
        caption=caption,
        label='fig:' + fig_name,
        height=None,
        width='\\textwidth',
        filename=fig_dir + '/figure_' + fig_name + '.tex'
    )

for fig_name, caption in screenshots:
    at.latex.make_figure(
        image_filename='screenshots/' + fig_name + '.png',
        caption=caption,
        label='fig:' + fig_name,
        height='5cm',
        filename=fig_dir + '/figure_' + fig_name + '_slides.tex'
    )